ConvNeXt-Tiny experiment V1

First `ConvNeXt-Tiny` baseline for cassava classification, using the stronger `ResNet_V15` training flow: `AdamW`, `weight_decay`, early stopping, best-checkpoint restore, and `.pt` export.


1. Libary and device setup

In [1]:
import copy
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms, models
from collections import Counter

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

In [2]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


2. Class count and Disease name

In [3]:
COMPETITION_DATA_DIR = Path("/kaggle/input/competitions/cassava-leaf-disease-classification")
WEIGHTS_PATH = Path("/kaggle/input/datasets/lawhan/convnext-pretrained-weights/convnext_tiny-983f1562.pth")  # Update this manually to match your attached Kaggle input

TRAIN_CSV_PATH = COMPETITION_DATA_DIR / "train.csv"
LABEL_MAP_PATH = COMPETITION_DATA_DIR / "label_num_to_disease_map.json"
TRAIN_IMAGE_DIR = COMPETITION_DATA_DIR / "train_images"
TEST_CSV_PATH = COMPETITION_DATA_DIR / "sample_submission.csv"
TEST_IMAGE_DIR = COMPETITION_DATA_DIR / "test_images"

with open(LABEL_MAP_PATH, "r") as f:
    label_num_to_disease_map = json.load(f)

train_df = pd.read_csv(TRAIN_CSV_PATH)
print(train_df.head())


         image_id  label
0  1000015157.jpg      0
1  1000201771.jpg      3
2   100042118.jpg      1
3  1000723321.jpg      1
4  1000812911.jpg      3


3. Train and Split Validation and Distribution

In [4]:
# Split dataset (80% train, 20% validation)
train_df_split, valid_df_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=42
)

# Count labels
train_counts = Counter(train_df_split["label"])
valid_counts = Counter(valid_df_split["label"])

print("Training set class counts:\n")
for class_idx, count in sorted(train_counts.items()):
    print(f"{label_num_to_disease_map[str(class_idx)]}: {count}")

print("\nValidation set class counts:\n")
for class_idx, count in sorted(valid_counts.items()):
    print(f"{label_num_to_disease_map[str(class_idx)]}: {count}")



Training set class counts:

Cassava Bacterial Blight (CBB): 870
Cassava Brown Streak Disease (CBSD): 1751
Cassava Green Mottle (CGM): 1909
Cassava Mosaic Disease (CMD): 10526
Healthy: 2061

Validation set class counts:

Cassava Bacterial Blight (CBB): 217
Cassava Brown Streak Disease (CBSD): 438
Cassava Green Mottle (CGM): 477
Cassava Mosaic Disease (CMD): 2632
Healthy: 516


4. ConvNeXt-Tiny setup (offline Kaggle input weights)


Attach the Kaggle Dataset input that contains `convnext_tiny-983f1562.pth`, then manually set `WEIGHTS_PATH` to the exact file location before rerunning this notebook.

This notebook expects the official TorchVision ConvNeXt-Tiny checkpoint or a directory containing it.


In [5]:
from torchvision import models

def resolve_weights_path(weights_path=WEIGHTS_PATH):
    weights_path = Path(weights_path)

    if weights_path.is_dir():
        candidate_patterns = [
            "convnext_tiny-983f1562.pth",
            "convnext_tiny*.pth",
            "*.pth",
        ]
        for pattern in candidate_patterns:
            candidates = sorted(weights_path.rglob(pattern))
            if candidates:
                resolved_path = candidates[0]
                print(f"Resolved checkpoint inside directory: {resolved_path}")
                return resolved_path
        raise FileNotFoundError(
            f"No ConvNeXt-Tiny checkpoint was found under directory: {weights_path}"
        )

    if not weights_path.exists():
        raise FileNotFoundError(f"Offline checkpoint not found: {weights_path}")

    return weights_path


def build_model(num_classes=5, weights_path=WEIGHTS_PATH, dropout_p=0.3):
    resolved_weights_path = resolve_weights_path(weights_path)

    model = models.convnext_tiny(weights=None)
    state_dict = torch.load(resolved_weights_path, map_location="cpu")
    model.load_state_dict(state_dict)

    feature_dim = model.classifier[-1].in_features
    model.classifier = nn.Sequential(
        model.classifier[0],
        model.classifier[1],
        nn.Dropout(p=dropout_p),
        nn.Linear(feature_dim, num_classes),
    )
    return model


In [6]:
learning_rate = 3e-4
weight_decay = 1e-4
dropout_p = 0.3
label_smoothing = 0.1
batch_size = 16
image_resize = 236
image_crop = 224

model = build_model(dropout_p=dropout_p).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)


5. Creating Dataset Class

In [7]:
class CassavaDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image_id"]
        label = self.df.iloc[idx]["label"]

        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

6. Train and evaluation transforms (ConvNeXt-Tiny default-style preprocessing)


In [8]:
train_transform = transforms.Compose([
    transforms.Resize((image_resize, image_resize)),
    transforms.RandomCrop((image_crop, image_crop)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((image_resize, image_resize)),
    transforms.CenterCrop((image_crop, image_crop)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


7. Creating Train and Valid dataset

In [9]:
train_dataset = CassavaDataset(
    df=train_df_split,
    image_dir=TRAIN_IMAGE_DIR,
    transform=train_transform
)

valid_dataset = CassavaDataset(
    df=valid_df_split,
    image_dir=TRAIN_IMAGE_DIR,
    transform=eval_transform
)


In [10]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)


8. Training ConvNeXt-Tiny model


In [11]:
num_epochs = 10
early_stop_patience = 3
epochs_without_improvement = 0
history = []
best_val_accuracy = 0.0
best_model_state = copy.deepcopy(model.state_dict())
best_epoch = 0
best_checkpoint_path = Path("best_convnext_tiny_v18_best.pth")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_accuracy = 100 * train_correct / train_total
    avg_train_loss = running_loss / train_total

    model.eval()
    valid_running_loss = 0.0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            _, predicted = torch.max(outputs, 1)

            valid_running_loss += loss.item() * labels.size(0)
            valid_total += labels.size(0)
            valid_correct += (predicted == labels).sum().item()

    valid_accuracy = 100 * valid_correct / valid_total
    avg_valid_loss = valid_running_loss / valid_total

    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "train_accuracy": train_accuracy,
            "valid_loss": avg_valid_loss,
            "valid_accuracy": valid_accuracy,
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
    )

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Training Loss: {avg_train_loss:.4f}")
    print(f"Training Accuracy: {train_accuracy:.2f}%")
    print(f"Validation Loss: {avg_valid_loss:.4f}")
    print(f"Validation Accuracy: {valid_accuracy:.2f}%")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}\n")

    if valid_accuracy > best_val_accuracy:
        best_val_accuracy = valid_accuracy
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    scheduler.step()

    if epochs_without_improvement >= early_stop_patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

history_df = pd.DataFrame(history)
model.load_state_dict(best_model_state)
torch.save(best_model_state, best_checkpoint_path)

print(f"Best validation accuracy: {best_val_accuracy:.2f}%")
print(f"Best epoch: {best_epoch}")
print(f"Saved {best_checkpoint_path}")
history_df


Epoch [1/10]
Training Loss: 0.7858
Training Accuracy: 81.19%
Validation Loss: 0.7003
Validation Accuracy: 85.33%
Learning Rate: 0.000300

Epoch [2/10]
Training Loss: 0.7158
Training Accuracy: 84.20%
Validation Loss: 0.6999
Validation Accuracy: 85.05%
Learning Rate: 0.000300

Epoch [3/10]
Training Loss: 0.6310
Training Accuracy: 88.54%
Validation Loss: 0.6713
Validation Accuracy: 86.82%
Learning Rate: 0.000150

Epoch [4/10]
Training Loss: 0.6097
Training Accuracy: 89.67%
Validation Loss: 0.6605
Validation Accuracy: 86.94%
Learning Rate: 0.000150

Epoch [5/10]
Training Loss: 0.5548
Training Accuracy: 92.35%
Validation Loss: 0.6744
Validation Accuracy: 86.73%
Learning Rate: 0.000075

Epoch [6/10]
Training Loss: 0.5281
Training Accuracy: 93.68%
Validation Loss: 0.7144
Validation Accuracy: 85.91%
Learning Rate: 0.000075

Epoch [7/10]
Training Loss: 0.4855
Training Accuracy: 95.89%
Validation Loss: 0.6834
Validation Accuracy: 87.29%
Learning Rate: 0.000037

Epoch [8/10]
Training Loss: 0.4691

,epoch,train_loss,train_accuracy,valid_loss,valid_accuracy,learning_rate
0,1,0.785822,81.194134,0.700321,85.327103,0.000300
1,2,0.715814,84.202839,0.699909,85.046729,0.000300
2,3,0.630950,88.543553,0.671276,86.822430,0.000150
3,4,0.609673,89.665245,0.660535,86.939252,0.000150
4,5,0.554816,92.346790,0.674350,86.728972,0.000075
5,6,0.528141,93.684641,0.714420,85.911215,0.000075
6,7,0.485489,95.887130,0.683403,87.289720,0.000037
7,8,0.469146,96.617398,0.694039,87.149533,0.000037
8,9,0.448300,97.645615,0.699398,87.523364,0.000019
9,10,0.439660,97.966933,0.710497,87.313084,0.000019


8b. Export best model bundle to `.pt`

This cell saves the best ConvNeXt-Tiny checkpoint plus lightweight metadata into a reusable `.pt` file.


In [12]:
best_checkpoint_path = Path("best_convnext_tiny_v18_best.pth")
best_model_pt_path = Path("best_convnext_tiny_v18_model.pt")

if "best_model_state" in globals():
    export_state_dict = best_model_state
else:
    export_state_dict = torch.load(best_checkpoint_path, map_location="cpu")

model_pt_bundle = {
    "model_name": "convnext_tiny",
    "num_classes": 5,
    "image_resize": image_resize,
    "image_crop": image_crop,
    "label_smoothing": label_smoothing,
    "dropout_p": dropout_p,
    "optimizer": "AdamW",
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "batch_size": batch_size,
    "best_epoch": best_epoch if "best_epoch" in globals() else None,
    "best_val_accuracy": best_val_accuracy if "best_val_accuracy" in globals() else None,
    "state_dict": export_state_dict,
}

torch.save(model_pt_bundle, best_model_pt_path)
print(f"Saved {best_model_pt_path.resolve()}")
print(f"Bundle keys: {list(model_pt_bundle.keys())}")


Saved /kaggle/working/best_convnext_tiny_v18_model.pt
Bundle keys: ['model_name', 'num_classes', 'image_resize', 'image_crop', 'label_smoothing', 'dropout_p', 'optimizer', 'learning_rate', 'weight_decay', 'batch_size', 'best_epoch', 'best_val_accuracy', 'state_dict']


8c. Reload the exported `.pt` bundle

Use this optional cell to confirm the saved file and inspect its metadata without rebuilding the model first.


In [13]:
loaded_model_pt_bundle = torch.load(best_model_pt_path, map_location="cpu")
loaded_model_summary = {
    key: value for key, value in loaded_model_pt_bundle.items() if key != "state_dict"
}
loaded_model_summary


{'model_name': 'convnext_tiny',
 'num_classes': 5,
 'image_resize': 236,
 'image_crop': 224,
 'label_smoothing': 0.1,
 'dropout_p': 0.3,
 'optimizer': 'AdamW',
 'learning_rate': 0.0003,
 'weight_decay': 0.0001,
 'batch_size': 16,
 'best_epoch': 9,
 'best_val_accuracy': 87.5233644859813}

In [ ]:
# Test set inference using the best validation checkpoint
test_df = pd.read_csv(TEST_CSV_PATH)
test_image_dir = TEST_IMAGE_DIR


class CassavaTestDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image_id"]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, img_name


test_dataset = CassavaTestDataset(test_df, test_image_dir, transform=eval_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model.eval()
preds = []
ids = []
with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        preds.extend(predicted.cpu().numpy().tolist())
        ids.extend(image_ids)

submission = pd.DataFrame({"image_id": ids, "label": preds})
submission_path = Path("submission.csv")
submission.to_csv(submission_path, index=False)
print(f"Saved {submission_path}")


Saved submission_convnext_tiny_v18.csv
